In [1]:
%pip install langchain===0.3.25 | tail -n 1
%pip install langchain-openai===0.3.19 | tail -n 1

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

In [1]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [4]:
@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.
    
    Args:
        a (int): first integer to be added
        b (int): second integer to be added

    Return:
        int: sum of a and b
    """
    return a + b

In [5]:
tools = [add]

llm_with_tools = llm.bind_tools(tools)

In [6]:
@tool
def subtract(a: int, b:int) -> int:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: int, b:int) -> int:
    """Multiply a and b."""
    return a * b

In [7]:
tool_map = {
    "add": add, 
    "subtract": subtract,
    "multiply": multiply
}

input_ = {
    "a": 1,
    "b": 2
}

tool_map["add"].invoke(input_)

3

In [8]:
tools = [add, subtract, multiply]

llm_with_tools = llm.bind_tools(tools)

In [9]:
query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)]

In [10]:
response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

print(type(response_1))
#print(response_1)

<class 'langchain_core.messages.ai.AIMessage'>

In [11]:
tool_calls_1 = response_1.tool_calls

tool_1_name = tool_calls_1[0]["name"]
tool_1_args = tool_calls_1[0]["args"]
tool_call_1_id = tool_calls_1[0]["id"]

print(f'tool name:\n{tool_1_name}')
print(f'tool args:\n{tool_1_args}')
print(f'tool call ID:\n{tool_call_1_id}')

tool name:
add
tool args:
{'a': 3, 'b': 2}
tool call ID:
call_IA70uYrkHnnGosmDZj62NxAZ

In [12]:
tool_response = tool_map[tool_1_name].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)

print(tool_message)

content='5' tool_call_id='call_IA70uYrkHnnGosmDZj62NxAZ'

In [13]:
chat_history.append(tool_message)

In [14]:
answer = llm_with_tools.invoke(chat_history)
print(type(answer))
print(answer.content)

<class 'langchain_core.messages.ai.AIMessage'>
3 + 2 equals 5.

In [15]:
class ToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        # Step 1: Initial user message
        chat_history = [HumanMessage(content=query)]

        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        if not response.tool_calls:
            return response.contet # Direct response, no tool needed
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 4: Call tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)

        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message])

        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content

In [16]:
my_agent = ToolCallingAgent(llm)

print(my_agent.run("one plus 2"))

print(my_agent.run("one - 2"))

print(my_agent.run("three times two"))

One plus two equals three.
The result of \( 1 - 2 \) is \(-1\).
Three times two equals 6.

In [20]:
@tool
def calculate_tip(total_amount: int, tip_percent:int) -> float:
    """Calculate percent to tip for given sum of money."""
    return total_amount * (tip_percent / 100)

In [21]:
tool_map = {
    "calculate_tip": calculate_tip
}

input_ = {
    "total_amount": 100,
    "tip_percent": 15
}

tool_map["calculate_tip"].invoke(input_)

15.0

In [25]:
class TipToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        # Step 1: Initial user message
        chat_history = [HumanMessage(content=query)]

        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        if not response.tool_calls:
            return response.contet # Direct response, no tool needed
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 4: Call tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)

        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message])

        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content

In [24]:
tools = [calculate_tip]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
my_agent = TipToolCallingAgent(llm)

print(my_agent.run("How much should I tip on $60 at 20%?"))

print(my_agent.run("How much should I tip on $80 at 15%?"))
print(my_agent.run("How much should I tip on $100 at 10%?"))

You should tip $12 on a $60 bill at 20%.